# Transformer Attention 的 PyTorch Shape 实验

本 Notebook 配合《第 1 章补充｜读懂 Transformer 代码所需的 PyTorch 基础》使用。目标不是训练模型，而是逐格观察：

$$[B,S,H] \rightarrow [B,S,N,D] \rightarrow [B,N,S,D] \rightarrow [B,N,S,S] \rightarrow [B,S,H]$$

请按顺序运行每个单元格，并在进入下一格前先猜测输出 Shape。


## 0. 环境准备

需要 Python 3.10+ 和 PyTorch。若环境中没有 PyTorch，可以取消下一格第一行的注释，安装后重新启动 Kernel。


In [ ]:
# %pip install torch

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
print('PyTorch version:', torch.__version__)


## 1. 认识 Tensor 与 Shape

- `B`：Batch size
- `S`：Sequence length
- `H`：Hidden size
- `N`：注意力头数
- `D`：每个头的维度，满足 $H=N\times D$


In [ ]:
B, S, H, N = 2, 4, 8, 2
assert H % N == 0
D = H // N
x = torch.randn(B, S, H)

print('x.shape    =', x.shape)
print('x.size(0)  =', x.size(0), '# B')
print('x.size(1)  =', x.size(1), '# S')
print('x.size(-1) =', x.size(-1), '# H')
print('D = H / N  =', D)
print('dtype      =', x.dtype)
print('device     =', x.device)
print('numel      =', x.numel())


### 1.1 索引会不会保留维度？

`x[:, -1, :]` 使用整数索引，序列维消失；`x[:, [-1], :]` 使用列表索引，保留长度为 1 的序列维。


In [ ]:
print('x[:, -1, :].shape    =', x[:, -1, :].shape)
print('x[:, [-1], :].shape =', x[:, [-1], :].shape)
print('x[:, :2, :].shape    =', x[:, :2, :].shape)


## 2. Embedding：`[B,S] → [B,S,H]`

Embedding 的输入是整数 token id，典型 Shape 为 `[B,S]`，而不是 `[B,S,1]`。


In [ ]:
V = 20
token_ids = torch.tensor([[2, 5, 9, 3], [1, 7, 4, 6]])
embedding = nn.Embedding(num_embeddings=V, embedding_dim=H)
embedded = embedding(token_ids)

print('token_ids:', token_ids.shape, token_ids.dtype)
print('embedding weight:', embedding.weight.shape)
print('embedded:', embedded.shape)


## 3. 拆分多头：`view` 与 `transpose`

先将隐藏维 H 拆成 N×D，再交换序列维和头维。这一步只是改变张量的组织方式，还没有计算注意力。


In [ ]:
projected = torch.randn(B, S, H)
heads_before_transpose = projected.view(B, S, N, D)
heads = heads_before_transpose.transpose(1, 2)

print('projected [B,S,H]:', projected.shape)
print('view [B,S,N,D]:', heads_before_transpose.shape)
print('transpose [B,N,S,D]:', heads.shape)
print('is contiguous after transpose:', heads.is_contiguous())


### 3.1 合并多头为什么需要 `contiguous()`？

`transpose` 通常只改变 Shape 和 stride，底层内存顺序没有同步重排。先恢复 `[B,S,N,D]`，再调用 `contiguous()`，最后稳定地 `view(B,S,H)`。


In [ ]:
heads_back = heads.transpose(1, 2)
merged = heads_back.contiguous().view(B, S, H)

print('transpose back [B,S,N,D]:', heads_back.shape)
print('contiguous before copy?:', heads_back.is_contiguous())
print('merged [B,S,H]:', merged.shape)
print('values preserved:', torch.allclose(projected, merged))


## 4. 构造 Causal Mask

未来位置使用负无穷。加到 Attention Score 后，这些位置经过 Softmax 会变为 0。Mask 使用 `[1,1,S,S]`，通过广播供所有 Batch 和 Head 共用。


In [ ]:
def causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    mask = torch.full(
        (1, 1, seq_len, seq_len),
        float('-inf'),
        device=device,
    )
    return torch.triu(mask, diagonal=1)

mask = causal_mask(S, x.device)
print('mask.shape =', mask.shape)
print(mask[0, 0])


## 5. 手写 Scaled Dot-Product Attention

$$O=\operatorname{softmax}\left(\frac{QK^T}{\sqrt D}+M\right)V$$

Q、K 最后一维必须相同；K、V 的序列长度必须相同。V 的最后一维理论上可以与 Q/K 不同。


In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    head_dim = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(head_dim)
    print('1. QK^T / sqrt(D):', scores.shape)

    if mask is not None:
        scores = scores + mask
        print('2. scores + mask:', scores.shape)

    probabilities = F.softmax(scores.float(), dim=-1).type_as(q)
    print('3. softmax:', probabilities.shape)

    output = torch.matmul(probabilities, v)
    print('4. probabilities @ V:', output.shape)
    return output, probabilities


In [ ]:
q = torch.randn(B, N, S, D)
k = torch.randn(B, N, S, D)
v = torch.randn(B, N, S, D)

attention_output, probabilities = scaled_dot_product_attention(q, k, v, mask)
row_sums = probabilities.sum(dim=-1)

print('\n每行 Softmax 之和的 Shape:', row_sums.shape)
print('是否都约等于 1:', torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-6))
print('第 0 个样本、第 0 个头的权重:\n', probabilities[0, 0])


观察上格输出：

- Score 的 Shape 是 `[B,N,S_q,S_kv]`；
- `dim=-1` 沿所有 Key 位置归一化；
- 权重矩阵主对角线上方应为 0；
- 每一行的权重和应约等于 1。


## 6. 拼成完整的多头自注意力

下面把线性投影、拆头、Attention、合头和输出投影串起来。留意：`nn.Linear` 只改变最后一维。


In [ ]:
class ShapeMultiHeadAttention(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int):
        super().__init__()
        assert hidden_size % num_heads == 0
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.wq = nn.Linear(hidden_size, hidden_size, bias=False)
        self.wk = nn.Linear(hidden_size, hidden_size, bias=False)
        self.wv = nn.Linear(hidden_size, hidden_size, bias=False)
        self.wo = nn.Linear(hidden_size, hidden_size, bias=False)

    def split_heads(self, tensor):
        batch_size, seq_len, _ = tensor.shape
        tensor = tensor.view(batch_size, seq_len, self.num_heads, self.head_dim)
        return tensor.transpose(1, 2)

    def merge_heads(self, tensor):
        batch_size, _, seq_len, _ = tensor.shape
        return tensor.transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.hidden_size
        )

    def forward(self, tensor):
        print('input:', tensor.shape)
        q = self.split_heads(self.wq(tensor))
        k = self.split_heads(self.wk(tensor))
        v = self.split_heads(self.wv(tensor))
        print('Q/K/V after split:', q.shape)

        mask = causal_mask(tensor.size(1), tensor.device)
        attended, probabilities = scaled_dot_product_attention(q, k, v, mask)
        merged = self.merge_heads(attended)
        print('merged:', merged.shape)
        output = self.wo(merged)
        print('output:', output.shape)
        return output, probabilities


In [ ]:
mha = ShapeMultiHeadAttention(hidden_size=H, num_heads=N)
mha_output, mha_probabilities = mha(embedded)
print('\n输入与输出 Shape 相同:', embedded.shape == mha_output.shape)


## 7. Decode：`S_q` 可以与 `S_kv` 不同

教材版 MHA 使用同一个 `seqlen` reshape Q、K、V，只覆盖等长情况。Decode 使用 KV Cache 时，当前 Query 通常只有 1 个 token，而 K/V 包含历史。


In [ ]:
S_q, S_kv = 1, 6
q_decode = torch.randn(B, N, S_q, D)
k_cache = torch.randn(B, N, S_kv, D)
v_cache = torch.randn(B, N, S_kv, D)

# 当前 Query 可以读取包括当前位置在内的全部缓存，此处不加方形因果 Mask。
decode_output, decode_probabilities = scaled_dot_product_attention(
    q_decode, k_cache, v_cache
)

print('Q:', q_decode.shape)
print('K/V cache:', k_cache.shape)
print('attention weights:', decode_probabilities.shape)
print('decode output:', decode_output.shape)


这格最值得记住：

$$[B,N,1,D]\times[B,N,D,S_{kv}]\rightarrow[B,N,1,S_{kv}]$$

这是后续理解 FIA Decode、KV Cache、PA 和 GQA 的 Shape 起点。


## 8. 动手练习

请修改参数并重新运行相关单元格：

1. 将 `B,S,H,N` 改为 `1,8,16,4`，写出每一步 Shape；
2. 将 `S_q,S_kv` 改为 `1,128`，观察 Decode Score；
3. 将 Softmax 的 `dim=-1` 改为 `dim=-2`，比较每个维度的求和结果；
4. 暂时移除 Causal Mask，观察未来位置是否获得非零权重；
5. 删除 `contiguous()` 后执行合头，记录是否报错以及原因。

做完后，应能解释：view 只是拆出 Head；transpose 组织批量矩阵；Score 最后一维对应 Key；Decode 的 `S_q` 与 `S_kv` 可以不同；Shape、stride、dtype 都会影响算子实现。
